# Leaflet cluster map of talk and poster locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` and `_posters/` directories contain `.md` files of all your talks and posters. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and generates an interactive map with color-coded markers (red for talks, blue for posters).

In [ ]:
# Start by installing the dependencies
!pip install python-frontmatter --upgrade
import frontmatter
import glob
import json
import time
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable

Iywidgets and ipyleaflet support disabled. You must be in a Jupyter notebook to use this feature.
Error raised:
No module named 'ipyleaflet'
Check that you have enabled ipyleaflet in Jupyter with:
    jupyter nbextension enable --py ipyleaflet


In [ ]:
# Collect the Markdown files from both talks and posters
g = glob.glob("_talks/*.md") + glob.glob("_posters/*.md")

In [ ]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_data = []  # List of dicts: {description, lat, lon, type, color}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [ ]:
# Perform geolocation
def is_virtual_location(value):
    if not value:
        return False
    lowered = value.lower()
    return "visioconf" in lowered or "visio" in lowered or "online" in lowered or "zoom" in lowered

def geocode_with_fallback(primary, fallback):
    for query in (primary, fallback):
        if not query:
            continue
        try:
            result = geocoder.geocode(query, timeout=TIMEOUT)
        except (GeocoderTimedOut, GeocoderUnavailable, ValueError) as ex:
            print(f"Error: geocode failed on input {query} with message {ex}")
            result = None
        time.sleep(1)
        if result:
            return result, query
    return None, None

for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Only show items explicitly enabled for the map
    if not data.get('showonmap', False):
        continue

    # Determine type (talk or poster) and color
    item_type = data.get('collection', 'talk')  # 'talks' or 'posters'
    if 'poster' in item_type:
        item_type = 'poster'
        color = '#2E5CA6'  # Blue for posters
    else:
        item_type = 'talk'
        color = '#D62828'  # Red for talks

    # Prepare the description
    title = data.get('title', '').strip()
    location = data.get('location', '').strip()
    if is_virtual_location(location):
        location = ""
    if location:
        description = f"{title}<br />{location}"
    else:
        description = title

    # Geocode the location and report the status
    result, used_query = geocode_with_fallback(location, location)
    if not result:
        print(f"Warning: geocode returned no result for {description}")
        continue
    
    location_data.append({
        'description': description,
        'latitude': result.latitude,
        'longitude': result.longitude,
        'type': item_type,
        'color': color
    })
    print(description, f"(geocoded from: {used_query})", f"[{item_type}]", result)

How to work with complex geometries in PINNs ?<br />Strasbourg, France; (Visioconférence) Salle de visioconférence, Rue Jaboulay, Jean-Macé, Lyon 7e Arrondissement, Lyon, Métropole de Lyon, Rhône, Auvergne-Rhône-Alpes, France métropolitaine, 69007, France
Combining Finite Element Methods and Neural Networks to Solve Elliptic Problems on 2D Geometries<br />Paris, France; Arts et Métiers – ENSAM, Paris, France None
Development of hybrid finite element/neural network methods to help create digital surgical twins<br />Strasbourg, France; Explora Building, Inria, Strasbourg, France None
Enriching continuous Lagrange finite element approximation spaces using neural networks<br />Montreal, Canada; McGill University, Montreal, Canada McGill University, 845, Rue Sherbrooke Ouest, Ville-Marie, Montréal, Agglomération de Montréal, Montréal (région administrative), Québec, H3A 3P8, Canada
Development of hybrid finite element/neural network methods to help create digital surgical twins<br />Strasbo

In [ ]:
# Save the map data as org-locations.js
import os
os.makedirs('talkmap', exist_ok=True)

with open('talkmap/org-locations.js', 'w', encoding='utf-8') as f:
    f.write('var addressPoints = [\n')
    for i, item in enumerate(location_data):
        comma = ',' if i < len(location_data) - 1 else ''
        f.write(f'  {{\n')
        f.write(f'    "description": "{item["description"]}",\n')
        f.write(f'    "latitude": {item["latitude"]},\n')
        f.write(f'    "longitude": {item["longitude"]},\n')
        f.write(f'    "type": "{item["type"]}",\n')
        f.write(f'    "color": "{item["color"]}"\n')
        f.write(f'  }}{comma}\n')
    f.write('];\n')

print(f"\nGenerated talkmap/org-locations.js with {len(location_data)} points")
print(f"  - Talks (red): {sum(1 for x in location_data if x['type'] == 'talk')}")
print(f"  - Posters (blue): {sum(1 for x in location_data if x['type'] == 'poster')}")

{'How to work with complex geometries in PINNs ?<br />Strasbourg, France; (Visioconférence)': Location(Salle de visioconférence, Rue Jaboulay, Jean-Macé, Lyon 7e Arrondissement, Lyon, Métropole de Lyon, Rhône, Auvergne-Rhône-Alpes, France métropolitaine, 69007, France, (45.7490315, 4.8366913, 0.0)),
 'Combining Finite Element Methods and Neural Networks to Solve Elliptic Problems on 2D Geometries<br />Paris, France; Arts et Métiers – ENSAM, Paris, France': None,
 'Development of hybrid finite element/neural network methods to help create digital surgical twins<br />Strasbourg, France; Explora Building, Inria, Strasbourg, France': None,
 'Enriching continuous Lagrange finite element approximation spaces using neural networks<br />Montreal, Canada; McGill University, Montreal, Canada': Location(McGill University, 845, Rue Sherbrooke Ouest, Ville-Marie, Montréal, Agglomération de Montréal, Montréal (région administrative), Québec, H3A 3P8, Canada, (45.5068861, -73.5787118, 0.0)),
 'Mesh-b

In [5]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)

'Written map to talkmap/'